# Homework: Multimodal Models - Image Captioning and Visual Question Answering

**Goal**

In this assignment, you will experiment with multimodal models that combine vision and language.

You will use a pretrained model to:

1. Generate a caption for an image.
2. Answer questions about an image.

You will also analyze the model's strengths, weaknesses, and possible failure cases.

## Task 1 - Image Captioning (30 Points)

**Description**

Given an input image, use a pretrained multimodal model to generate a textual description of the image.

Example:

**Input**: An image of a dog running in a park.

**Output**: A dog is running on grass in a park.

**Requirements**

Use at least 5 different images.

The images should include:

* At least one image with people
* At least one image with animals
* At least one indoor scene
* At least one outdoor scene
* At least one image with multiple objects

You may use your own images or images from an open dataset.

**Suggested Models**

You may use one of the following:

* Salesforce/blip-image-captioning-base
* Salesforce/blip-image-captioning-large
* microsoft/git-base
* Any other image-captioning model from HuggingFace

**Deliverables**

1. Present your images and model captions clearly

    Display each image together with the caption generated by the model.

2. Analyze model failures

    Choose at least 2 examples where the model made a mistake or gave an incomplete caption.

    For each example, explain:

    * What the model predicted
    * What the correct description should be
    * Why the model may have failed

    For example :

    "The model described the image as “a person sitting at a table,” but it missed the laptop and the coffee cup.
    This may have happened because the person was the most visually dominant object, while the smaller objects were less noticeable."


In [1]:
# Installs the dependencies this notebook needs.
%pip install -q "transformers>=4.40" pillow matplotlib requests

import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device}")

Note: you may need to restart the kernel to use updated packages.
torch 2.2.2+cu121 | device: cpu


In [3]:
# Use the *base* checkpoint first: a decent model, which is not too heavy.
import os
from transformers import logging as hf_logging

# Silence redundent warnings.
hf_logging.set_verbosity_error()

MODEL_NAME = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME, use_safetensors=True, token=os.environ["HF_TOKEN"])
model = model.to(device).eval()


def generate_caption(image: Image.Image) -> str:
    """Generate a caption for a single PIL image."""
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, num_beams=5)
    return processor.decode(out[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

In [ ]:
# My Images:

CAPTION = "caption"
URL = "url"

images_spec = [
    {CAPTION: "two dogs playing in a living room", URL: "/home/yaron/WorkEnv/data/nebius/pics/lea_and_kiki.png"},
    {CAPTION: "shibley standing on my knees", URL: "/home/yaron/WorkEnv/data/nebius/pics/shibley_on_my_knees.png"},
    {CAPTION: "yossie tzabari holding a flag", URL: "/home/yaron/WorkEnv/data/nebius/pics/yossie_tzabari.png"},
    {CAPTION: "hadar explaining to tv talents", URL: "/home/yaron/WorkEnv/data/nebius/pics/hadar_explaining.png"},
    {CAPTION: "mika joining the army", URL: "/home/yaron/WorkEnv/data/nebius/pics/mika_mitgayeset.png"},
]


def load_image(url: str) -> Image.Image:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert("RGB")


# To use local images, comment the loop above and do, e.g.:
# images_spec = [
#     {"category": "people", "image": Image.open("my_photos/people.jpg").convert("RGB")},
#     ...
# ]
for spec in images_spec:
    if "image" not in spec:
        spec["image"] = load_image(spec["url"])

print(f"Loaded {len(images_spec)} images.")

In [ ]:
# --- Generate captions and display each image together with its caption ---
n = len(images_spec)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 6))
if n == 1:
    axes = [axes]

for ax, spec in zip(axes, images_spec):
    caption = generate_caption(spec["image"])
    spec["caption"] = caption  # keep for the failure-analysis section
    ax.imshow(spec["image"])
    ax.axis("off")
    ax.set_title(f"[{spec['category']}]\n{caption}", fontsize=11, wrap=True)

plt.tight_layout()
plt.show()

# Also print captions as text for easy reference
for spec in images_spec:
    print(f"- ({spec['category']}) {spec['caption']}")

## Task 2 - Visual Question Answering (30 points)

**Description**

Given an image and a natural-language question, use a pretrained multimodal model to answer the question.

Example:

**Input image**: A kitchen scene

**Question**: How many chairs are visible?

**Output**: Four.

**Requirements**

Use at least 3 images.

For each image, ask at least 3 questions.

Your questions should include:

* One object-recognition question
* One counting question
* One color or attribute question

Example questions:
* What animal is in the image?
* How many people are visible?
* What color is the car?

**Suggested Models**

You may use:

* Salesforce/blip-vqa-base
* Salesforce/blip-vqa-capfilt-large
* Any other VQA model from HuggingFace

**Deliverables**

1. Present your images, questions and model's answers clearly

    Display each image together with the Q&A generated by the model.

2. Analyze model failures

    Choose at least 2 examples where the model made a mistake or gave an incomplete caption.

    For each example, explain:

    * What the model predicted
    * What the correct description should be
    * Why the model may have failed

    **You are required to find examples where it fails and analyse why**


In [ ]:
# code goes here

## Task 3 - Personalized Face Recognition with a Celebrity Dataset (40 points)


**Goal**

Build a small personalized multimodal recognition system.

For this task, you will use a public celebrity face dataset rather than private photos of family or friends.

Although the same approach could easily be adapted to recognize your own friends, family members, or personal image collections, in this assignment you must use a celebrity dataset in order to avoid privacy concerns and to ensure a consistent evaluation setting across all students.

Students who are interested may later explore how the same techniques can be extended to build personalized multimodal recognition systems for recreational or educational purposes outside the scope of this course.

The system should learn to recognize a small set of known identities and answer questions such as:

* Who is in the image?
* Is this person Emma Watson or Leonardo DiCaprio?
* What is Emma doing or wearing?

Recommended datasets:

* CelebA — contains over 200K celebrity face images, 10,177 identities, and 40 attribute annotations. https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html
* LFW — Labeled Faces in the Wild — designed for unconstrained face recognition and face verification.
https://scikit-learn.org/0.19/datasets/labeled_faces.html
* You may also scrape or download images from the internet in order to build your celebrity dataset.

1. Choose 3-5 identities from a celebrity dataset. For each identity, collect at least 8 images.

2. split the images into:
   * Train/reference set: 5-6 images per identity
   * Test/query set: 2-3 images per identity

3. Use a pretrained model to convert each image into an embedding vector. Recommended model : CLIP

4. Compare a query image to the known identity images using cosine similarity

5. Use the predicted identity to answer simple personalized questions.

    Note:

    CLIP similarity predicts who the person is; a VQA/captioning model answers visual attribute/action questions about the image.

    Use this structure:
    1. CLIP similarity → Who is this?
    2. VQA model → What is the person wearing / doing / holding?
    3. Combined answer → “This is probably Emma Watson. She appears to be wearing a black dress.”

              Example:

              Question: Who is in the image?

              Answer: This is probably Emma Watson.

              Question: Is this Barack Obama?

              Answer: No, this is probably Taylor Swift.

              Question: What is Emma wearing?

              Answer: The model appears to show Emma wearing a black jacket.

6. Analyze when the system works and when it fails. Show images , Q&A and disscuess results.

In [ ]:
# code goes here